Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [52]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_inventory.csv")

data.head(10)

,product_id,gudang,stok_tersedia,last_update
0,P001,Gudang Bandung,40.0,15/06/2024
1,P001,GUDANG MEDAN,80.0,2024-07-01
2,P001,gudang makassar,0.0,2024-07-12 11:51:00
3,P002,Gudang Jakarta,10.0,2024-07-16 06:30:00
4,P002,Gudang Bandung,80.0,26/06/2024
5,P002,GUDANG MEDAN,10.0,2024-06-11
6,P003,Gudang Jakarta,NaN,2024-07-10
7,P003,GUDANG MEDAN,80.0,2024-07-06
8,P003,gudang makassar,0.0,12/07/2024
9,P004,Gudang Jakarta,25.0,2024-06-13 06:01:00


In [53]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 37
Kolom: ['product_id', 'gudang', 'stok_tersedia', 'last_update']
<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     37 non-null     str    
 1   gudang         37 non-null     str    
 2   stok_tersedia  34 non-null     float64
 3   last_update    37 non-null     str    
dtypes: float64(1), str(3)
memory usage: 1.3 KB


In [54]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
0


In [55]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
product_id       0
gudang           0
stok_tersedia    3
last_update      0
dtype: int64


In [56]:
data.nunique()

product_id       10
gudang            5
stok_tersedia     9
last_update      36
dtype: int64

In [57]:
# Nilai Negatif
stok_negatif = data[data['stok_tersedia'] < 0]
print(stok_negatif)

   product_id           gudang  stok_tersedia          last_update
11       P004   Gudang Bandung           -5.0  2024-06-09 01:58:00
13       P004  gudang makassar           -5.0           29/06/2024
16       P005     GUDANG MEDAN           -5.0  2024-06-07 17:03:00
18       P006   Gudang Jakarta           -5.0           05/06/2024
19       P006  Gudang Surabaya           -5.0           2024-07-15
20       P006   Gudang Bandung           -5.0  2024-07-03 07:44:00
33       P010   Gudang Bandung           -5.0  2024-07-15 10:05:00


Step 2: TRANSFORM - Bersihkan Data

In [58]:
# Tipe diskon: Semua huruf kecil
data['gudang'] = data['gudang'].str.strip().str.replace('_', ' ', regex=False).str.title()
# Status: Huruf pertama kapital
#data['gudang'] = data['gudang'].str.strip().str.capitalize()
print(data[['gudang']])

             gudang
0    Gudang Bandung
1      Gudang Medan
2   Gudang Makassar
3    Gudang Jakarta
4    Gudang Bandung
5      Gudang Medan
6    Gudang Jakarta
7      Gudang Medan
8   Gudang Makassar
9    Gudang Jakarta
10  Gudang Surabaya
11   Gudang Bandung
12     Gudang Medan
13  Gudang Makassar
14  Gudang Surabaya
15   Gudang Bandung
16     Gudang Medan
17  Gudang Makassar
18   Gudang Jakarta
19  Gudang Surabaya
20   Gudang Bandung
21     Gudang Medan
22  Gudang Makassar
23   Gudang Jakarta
24  Gudang Surabaya
25   Gudang Bandung
26   Gudang Jakarta
27   Gudang Bandung
28     Gudang Medan
29  Gudang Makassar
30   Gudang Jakarta
31  Gudang Makassar
32  Gudang Surabaya
33   Gudang Bandung
34     Gudang Medan
35  Gudang Makassar
36  Gudang Surabaya


In [59]:
# Mengisi missing value pada stok barang stok_tersedia dengan 0
# Mengisi missing value pada stok_tersedia menjadi 0.0
data['stok_tersedia'] = data['stok_tersedia'].fillna(0.0)
print(data['stok_tersedia'].isna().sum())

0


In [60]:
# Mengubah stok_tersedia bernilai negatif menjadi positif 
# Melihat data stok yang bernilai negatif
stok_negatif = data[data['stok_tersedia'] < 0]
display(stok_negatif)

,product_id,gudang,stok_tersedia,last_update
11,P004,Gudang Bandung,-5.0,2024-06-09 01:58:00
13,P004,Gudang Makassar,-5.0,29/06/2024
16,P005,Gudang Medan,-5.0,2024-06-07 17:03:00
18,P006,Gudang Jakarta,-5.0,05/06/2024
19,P006,Gudang Surabaya,-5.0,2024-07-15
20,P006,Gudang Bandung,-5.0,2024-07-03 07:44:00
33,P010,Gudang Bandung,-5.0,2024-07-15 10:05:00


In [61]:
# Mengubah stok_tersedia bernilai negatif menjadi positif
data['stok_tersedia'] = data['stok_tersedia'].abs()


In [62]:
# Mengubah format tanggal menjadi standar ISO 2024-06-27
# Mengubah berbagai format tanggal menjadi datetime
data['last_update'] = pd.to_datetime(
    data['last_update'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)
data['last_update'] = data['last_update'].dt.strftime('%Y-%m-%d')
print(data['last_update'].isna().sum())
data.head()

0


,product_id,gudang,stok_tersedia,last_update
0,P001,Gudang Bandung,40.0,2024-06-15
1,P001,Gudang Medan,80.0,2024-01-07
2,P001,Gudang Makassar,0.0,2024-12-07
3,P002,Gudang Jakarta,10.0,2024-07-16
4,P002,Gudang Bandung,80.0,2024-06-26


In [63]:
# Mengubah stok_tersedia menjadi nilai angka normal
(data[['stok_tersedia']].isna().sum())
data['stok_tersedia'] = data['stok_tersedia'].astype(int)
data.head(10)

,product_id,gudang,stok_tersedia,last_update
0,P001,Gudang Bandung,40,2024-06-15
1,P001,Gudang Medan,80,2024-01-07
2,P001,Gudang Makassar,0,2024-12-07
3,P002,Gudang Jakarta,10,2024-07-16
4,P002,Gudang Bandung,80,2024-06-26
5,P002,Gudang Medan,10,2024-11-06
6,P003,Gudang Jakarta,0,2024-10-07
7,P003,Gudang Medan,80,2024-06-07
8,P003,Gudang Makassar,0,2024-07-12
9,P004,Gudang Jakarta,25,2024-06-13


In [64]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
product_id       0
gudang           0
stok_tersedia    0
last_update      0
dtype: int64


In [65]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     37 non-null     str  
 1   gudang         37 non-null     str  
 2   stok_tersedia  37 non-null     int64
 3   last_update    37 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.3 KB


In [66]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/inventory_clean.csv",
    index=False,
    encoding="utf-8"
)
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/inventory_clean.csv")
data.head()

,product_id,gudang,stok_tersedia,last_update
0,P001,Gudang Bandung,40,2024-06-15
1,P001,Gudang Medan,80,2024-01-07
2,P001,Gudang Makassar,0,2024-12-07
3,P002,Gudang Jakarta,10,2024-07-16
4,P002,Gudang Bandung,80,2024-06-26
